# Noise collection

Collect a diverse sample of background audio from every cached embedding file (`<recording>.v2.pkl`, as written by the handler) and write it back to back into one wav, e.g. as a `--noise-wav` for `find`.

1. **Eligible vectors:** frames the embedding model classified as `NOISE` whose whole analysis window (2 hops) is covered only by NOISE windows, i.e. interior frames of a NOISE run of at least `MIN_RUN_FRAMES`.
2. **Sample:** pick `SAMPLE_FRACTION` of them with k-means++ seeding (D² sampling) over the embedding vectors -- only the seeding, no Lloyd iterations, no re-estimated centres. Frames far from everything already picked are likely to be picked next, so the sample covers the kinds of background instead of repeating the most common one. Centres are drawn in batches of `BATCH` with the same D² probabilities (the k-means|| variant) so it scales to millions of vectors; `BATCH = 1` is exact k-means++.
3. **Write:** each sampled vector writes its full window; overlapping windows are merged into one span so no audio is written twice. Spans are written back to back with a short fade, and a CSV lists every span.

In [6]:
import os
import glob
import time
import pickle
import itertools
import subprocess
import tempfile

import numpy as np
import pandas as pd
import soundfile as sf

In [2]:
# ---- configuration ----
REPO           = os.path.abspath('..')
EMBEDDINGS_DIR = os.path.join(REPO, 'motif', 'handler_state', 'motif_embeddings_cache')   # <recording>.<version>.pkl
EMBEDDINGS_VERSION = 'v2'
AUDIO_ROOT     = '/Users/dkohlsdorf/Documents/audio/wdp_data_at_google'   # searched recursively for .wav / .m4a

OUT_WAV = os.path.join(REPO, 'motif', 'handler_state', 'noise_collection.wav')
OUT_CSV = os.path.splitext(OUT_WAV)[0] + '_spans.csv'

NOISE_LABELS    = {'NOISE'}   # frame classifications counted as noise
MIN_RUN_FRAMES  = 5           # only sample from NOISE runs at least this long (one frame ~58 ms)
SAMPLE_FRACTION = 0.05        # fraction of eligible vectors to sample
BATCH           = 256         # centres drawn per D² round; 1 = exact k-means++ (slow)
SEED            = 0
WINDOW_HOPS     = 2           # analysis window per vector, in hops (5120 / 2560 samples)
FADE_S          = 0.005       # raised-cosine fade in/out per span, so cuts don't click; 0 = hard cuts
OUTPUT_SR       = None        # None = the embeddings' own sample rate
RECORDINGS      = None        # list of recording names to restrict to; None = all

## Eligible noise vectors

In [3]:
def find_files(root, name):
    return sorted(glob.glob(os.path.join(glob.escape(root), '**', name), recursive=True))


audio_index = {}
for ext in ('m4a', 'wav'):   # wav preferred over m4a
    for p in find_files(AUDIO_ROOT, '*.' + ext):
        audio_index[os.path.splitext(os.path.basename(p))[0]] = p

suffix = f'.{EMBEDDINGS_VERSION}.pkl'
X, rec_ids, frames, recordings, missing, rates = [], [], [], [], [], set()
for p in sorted(glob.glob(os.path.join(glob.escape(EMBEDDINGS_DIR), '*' + suffix))):
    recording = os.path.basename(p)[:-len(suffix)]
    if RECORDINGS is not None and recording not in RECORDINGS:
        continue
    if recording not in audio_index:
        missing.append(recording)
        continue
    with open(p, 'rb') as f:
        emb, classifications, step_samples, emb_sr = pickle.load(f)
    rates.add((step_samples, emb_sr))
    keep, frame = [], 0
    for is_noise, run in itertools.groupby(classifications, key=lambda c: c in NOISE_LABELS):
        n = len(list(run))
        if is_noise and n >= MIN_RUN_FRAMES:
            # interior frames only: frame f's window spans hops f .. f+WINDOW_HOPS-1, and the
            # windows touching those hops start at f-WINDOW_HOPS+1 .. f+WINDOW_HOPS-1
            margin = WINDOW_HOPS - 1
            keep.extend(range(frame + margin, frame + n - margin))
        frame += n
    keep = np.array(keep, dtype=int)
    X.append(np.asarray(emb, dtype=np.float32)[keep])
    rec_ids.append(np.full(len(keep), len(recordings)))
    frames.append(keep)
    recordings.append(recording)

if missing:
    print(f'WARNING: no audio under {AUDIO_ROOT} for {len(missing)} recordings, skipped: {missing}')
assert len(rates) == 1, f'embeddings at several hop/rates {rates}'
step_samples, emb_sr = rates.pop()
OUTPUT_SR = OUTPUT_SR or emb_sr
hop_s = step_samples / emb_sr
X, rec_ids, frames = np.vstack(X), np.concatenate(rec_ids), np.concatenate(frames)
print(f'{len(X)} eligible noise vectors ({len(X) * hop_s / 3600:.1f} h of hops) from {len(recordings)} recordings')

1100401 eligible noise vectors (17.7 h of hops) from 75 recordings


## k-means++ sample

In [4]:
def kmeanspp_sample(X, k, batch, seed, block=1 << 18):
    # D² sampling: each round draws `batch` new centres with probability proportional to
    # the squared distance to the nearest centre so far, then updates those distances
    rng = np.random.default_rng(seed)
    n = len(X)
    sq = np.einsum('ij,ij->i', X, X)
    D = np.full(n, np.inf, dtype=np.float32)

    def update(idx):
        C = X[idx]
        sqC = sq[idx]
        for a in range(0, n, block):
            d = sq[a:a + block, None] - 2 * X[a:a + block] @ C.T + sqC[None, :]
            np.minimum(D[a:a + block], np.maximum(d, 0).min(axis=1), out=D[a:a + block])

    chosen = [rng.integers(n)]
    update(np.array(chosen))
    t, last = time.time(), 0
    while len(chosen) < k:
        p = D.astype(np.float64)
        total = p.sum()
        if total == 0:          # everything left duplicates a centre
            break
        b = min(batch, k - len(chosen), int((p > 0).sum()))
        idx = rng.choice(n, size=b, replace=False, p=p / total)
        chosen.extend(idx.tolist())
        update(idx)
        if len(chosen) - last >= k // 10:
            last = len(chosen)
            print(f'  {len(chosen)}/{k} sampled, {time.time() - t:.0f}s')
    return np.array(chosen)


k = int(round(SAMPLE_FRACTION * len(X)))
chosen = kmeanspp_sample(X, k, BATCH, SEED)
print(f'sampled {len(chosen)} of {len(X)} vectors')
pd.Series(np.bincount(rec_ids[chosen], minlength=len(recordings)), index=recordings, name='sampled').to_frame() \
    .assign(eligible=np.bincount(rec_ids, minlength=len(recordings))).sort_values('sampled', ascending=False).head(15)

  5633/55020 sampled, 19s
  11265/55020 sampled, 36s
  16897/55020 sampled, 53s
  22529/55020 sampled, 70s
  28161/55020 sampled, 87s
  33793/55020 sampled, 111s
  39425/55020 sampled, 134s
  45057/55020 sampled, 152s
  50689/55020 sampled, 171s
sampled 55020 of 1100401 vectors


,sampled,eligible
07312401_EAGLE_TASCAM,2437,35465
07282101_Manta,2126,44925
07052101_Manta,1853,40094
07252502_SKATE_TASCAM (jump 2 - DUB setting),1851,53219
06042102_Manta,1839,38114
07012101_Manta,1632,34040
07032101_Manta,1631,35357
05062102_MANTA,1427,27546
06212101_MANTA,1239,24872
07292101_Manta,1235,27329


## Write the collection

In [5]:
# each sampled vector covers hops f .. f+WINDOW_HOPS-1; merge overlapping/adjacent ones into spans
rows = []
for r in np.unique(rec_ids[chosen]):
    hops = np.unique((frames[chosen[rec_ids[chosen] == r]][:, None] + np.arange(WINDOW_HOPS)).ravel())
    breaks = np.flatnonzero(np.diff(hops) > 1)
    for a, b in zip(np.r_[0, breaks + 1], np.r_[breaks, len(hops) - 1]):
        rows.append(dict(recording=recordings[r], start_s=hops[a] * hop_s, stop_s=(hops[b] + 1) * hop_s,
                         n_vectors=int(np.isin(frames[chosen[rec_ids[chosen] == r]], hops[a:b + 1]).sum())))
spans = pd.DataFrame(rows)
spans['duration_s'] = spans.stop_s - spans.start_s
bytes_out = spans.duration_s.sum() * OUTPUT_SR * 2
print(f'{len(spans)} spans, {spans.duration_s.sum() / 3600:.2f} h, ~{bytes_out / 1e9:.2f} GB at {OUTPUT_SR} Hz 16-bit')


def decode_mono(path, sr, tmp_dir):
    # whole recording -> mono wav at `sr` via ffmpeg, then read back
    out = os.path.join(tmp_dir, 'decoded.wav')
    subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', path, '-ac', '1', '-ar', str(sr),
                    '-c:a', 'pcm_s16le', out], check=True)
    x, _ = sf.read(out, dtype='float32')
    os.remove(out)
    return x


def fade(clip, n):
    if n > 0 and len(clip) > 2 * n:
        ramp = 0.5 - 0.5 * np.cos(np.linspace(0, np.pi, n, dtype=np.float32))
        clip[:n] *= ramp
        clip[-n:] *= ramp[::-1]
    return clip


n_fade = int(round(FADE_S * OUTPUT_SR))
fmt = 'WAV' if bytes_out < 3.9e9 else 'RF64'   # plain WAV tops out at 4 GB
os.makedirs(os.path.dirname(OUT_WAV), exist_ok=True)
out_start, written = [], 0
with sf.SoundFile(OUT_WAV, 'w', samplerate=OUTPUT_SR, channels=1, subtype='PCM_16', format=fmt) as out, \
        tempfile.TemporaryDirectory() as tmp:
    groups = spans.groupby('recording', sort=False)
    for i, (recording, g) in enumerate(groups):
        x = decode_mono(audio_index[recording], OUTPUT_SR, tmp)
        for _, s in g.iterrows():
            a, b = int(round(s.start_s * OUTPUT_SR)), min(len(x), int(round(s.stop_s * OUTPUT_SR)))
            out_start.append(written / OUTPUT_SR)
            if b > a:
                out.write(fade(x[a:b].copy(), n_fade))
                written += b - a
        print(f'[{i + 1}/{groups.ngroups}] {recording}: {len(g)} spans, total {written / OUTPUT_SR / 60:.1f} min')

spans['out_start_s'] = out_start
spans.to_csv(OUT_CSV, index=False)
print(f'wrote {written / OUTPUT_SR / 3600:.2f} h ({os.path.getsize(OUT_WAV) / 1e9:.2f} GB) to {OUT_WAV}\nspans: {OUT_CSV}')

49277 spans, 1.72 h, ~0.55 GB at 44100 Hz 16-bit
[1/75] 05062102_MANTA: 1286 spans, total 2.7 min
[2/75] 05072101_Manta: 112 spans, total 2.9 min
[3/75] 05082101_Manta: 633 spans, total 4.2 min
[4/75] 05102101_Manta: 832 spans, total 6.0 min
[5/75] 05102102_Manta: 121 spans, total 6.3 min
[6/75] 05102103_Manta: 327 spans, total 6.9 min
[7/75] 05102104_Manta: 225 spans, total 7.4 min
[8/75] 05112102_Manta: 964 spans, total 9.4 min
[9/75] 05222401_SKATE_TASCAM: 238 spans, total 10.0 min
[10/75] 05232401_SKATE_TASCAM: 1054 spans, total 12.2 min
[11/75] 05242301_SKATE: 212 spans, total 12.7 min
[12/75] 05262401_SKATE_TASCAM: 421 spans, total 13.5 min
[13/75] 05272401_SKATE_TASCAM: 811 spans, total 15.3 min
[14/75] 06022101_Manta: 511 spans, total 16.3 min
[15/75] 06022102_Manta: 181 spans, total 16.7 min
[16/75] 06042101_Manta: 398 spans, total 17.5 min
[17/75] 06042102_Manta: 1673 spans, total 21.0 min
[18/75] 06042103_Manta: 1051 spans, total 23.2 min
[19/75] 06042104_Manta: 671 spans, t